# Lab 09-13 · Compound and Wide-Bandgap Semiconductors

Companion to chapter [09-13](../../chapters/09-13-compound-and-wide-bandgap-semiconductors.md). Handout: [README.md](README.md).

In [1]:
import math
import numpy as np
from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q = 1.602176634e-19
E0 = 8.8541878128e-14
EPS_SI, EPS_SIC = 11.7 * E0, 9.96 * E0
an_si = lambda F: 3.318e5 * np.exp(-1.174e6 / F)
ap_si = lambda F: np.where(F < 5.3e5, 2e6 * np.exp(-1.97e6 / F), 5.6e5 * np.exp(-1.32e6 / F))
an_sic = lambda F: 1.76e8 * np.exp(-3.3e7 / F)
ap_sic = lambda F: 3.41e8 * np.exp(-2.5e7 / F)
mun = lambda N: 92 + 1268 / (1 + (N / 1.3e17) ** 0.91)

def ion_integral(Em, N, eps, an, ap):
    W = eps * Em / (Q * N)
    x = np.linspace(0, W, 20001)
    F = np.maximum(Em * (1 - x / W), 1e3)
    a, b = an(F), ap(F)
    inner = cumulative_trapezoid(a - b, x, initial=0)
    return trapezoid(a * np.exp(np.clip(-inner, -700, 700)), x)

def breakdown(N, eps, an, ap, lo, hi):
    Em = brentq(lambda E: ion_integral(E, N, eps, an, ap) - 1, lo, hi, xtol=1.0)
    W = eps * Em / (Q * N)
    return Em, W, Em * W / 2

SI = (EPS_SI, an_si, ap_si, 1.0e5, 1.5e6)
SIC = (EPS_SIC, an_sic, ap_sic, 1.2e6, 4e6)

## Part 1 — Breakdown against doping

*Predict first:* at 10¹⁶ cm⁻³, how many times higher is SiC's breakdown voltage?

In [2]:
Vs, Vc = breakdown(1e16, *SI)[2], breakdown(1e16, *SIC)[2]
print(f"V_B at 1e16: Si {Vs:.1f} V, SiC {Vc:.0f} V, ratio {Vc/Vs:.1f}")
assert abs(Vs - 55.6) < 0.05 and abs(Vc - 1996) < 0.5
Ns = np.logspace(14, 17, 16)
plt.loglog(Ns, [breakdown(n, *SI)[2] for n in Ns], label="Si")
NsC = np.logspace(15, 17, 11)
plt.loglog(NsC, [breakdown(n, *SIC)[2] for n in NsC], label="4H-SiC")
plt.xlabel("doping of the lightly doped side (cm^-3)"); plt.ylabel("V_B (V)"); plt.legend(); plt.show()

V_B at 1e16: Si 55.6 V, SiC 1996 V, ratio 35.9


/tmp/ipykernel_23788/3595067148.py:24: RuntimeWarning: overflow encountered in multiply
  return trapezoid(a * np.exp(np.clip(-inner, -700, 700)), x)
/home/user/.venv-ee/lib/python3.13/site-packages/scipy/integrate/_quadrature.py:154: RuntimeWarning: overflow encountered in add
  d * (y[tuple(slice1)] + y[tuple(slice2)]) / 2.0,


## Part 2 — Baliga's limit

Design the triangular-field drift layer for each voltage, then $R_{on,sp} = W/q\mu N$.

*Predict first:* does SiC's advantage grow or shrink from 600 V to 3300 V?

In [3]:
def design(V, mat, Nlo, Nhi, mu=None):
    N = brentq(lambda n: breakdown(n, *mat)[2] - V, Nlo, Nhi, rtol=1e-7)
    Em, W, _ = breakdown(N, *mat)
    m = mun(N) if mu is None else mu
    return N, Em, W, W / (Q * m * N)
res = {}
for V in (600, 1200, 3300):
    s = design(V, SI, 5e12, 2e15); c = design(V, SIC, 5e14, 3e17, 1000)
    res[V] = (s, c)
    print(f"{V:5d} V: Si N {s[0]:.3g}, W {s[2]*1e4:.1f} um, R {s[3]*1e3:.3g} mOhm cm2 | "
          f"SiC N {c[0]:.3g}, Ec {c[1]:.3g} V/cm, W {c[2]*1e4:.2f} um, R {c[3]*1e3:.3g} mOhm cm2 | ratio {s[3]/c[3]:.0f}")
r6, r12, r33 = (res[V][0][3] / res[V][1][3] for V in (600, 1200, 3300))
assert abs(r12 - 1386) < 1.5 and abs(r6 - 1242) < 1.5 and abs(r33 - 1585) < 1.5
assert abs(res[3300][1][1] - 2.56e6) < 5e3
Vs_ = np.array([400, 600, 900, 1200, 1700, 2500, 3300])
plt.loglog(Vs_, [design(V, SI, 5e12, 5e15)[3] * 1e3 for V in Vs_], "o-", label="Si")
plt.loglog(Vs_, [design(V, SIC, 5e14, 5e17, 1000)[3] * 1e3 for V in Vs_], "s-", label="4H-SiC")
plt.xlabel("V_B (V)"); plt.ylabel("ideal R_on,sp (mOhm cm^2)"); plt.legend(); plt.show()

/tmp/ipykernel_23788/3595067148.py:24: RuntimeWarning: overflow encountered in multiply
  return trapezoid(a * np.exp(np.clip(-inner, -700, 700)), x)


  600 V: Si N 3.22e+14, W 49.1 um, R 70.3 mOhm cm2 | SiC N 4.31e+16, Ec 3.07e+06 V/cm, W 3.91 um, R 0.0566 mOhm cm2 | ratio 1242
 1200 V: Si N 1.28e+14, W 110.1 um, R 395 mOhm cm2 | SiC N 1.85e+16, Ec 2.84e+06 V/cm, W 8.45 um, R 0.285 mOhm cm2 | ratio 1386


 3300 V: Si N 3.46e+13, W 351.1 um, R 4.66e+03 mOhm cm2 | SiC N 5.47e+15, Ec 2.56e+06 V/cm, W 25.76 um, R 2.94 mOhm cm2 | ratio 1585


## Part 3 — AlGaN/GaN polarization

*Predict first:* is the sheet density linear in $x$?

In [4]:
G = dict(a=3.189, e31=-0.35, e33=1.27, c13=106.0, c33=398.0, Psp=-0.029)
A = dict(a=3.112, e31=-0.50, e33=1.79, c13=108.0, c33=373.0, Psp=-0.081)
def sheet(x):
    p = {k: A[k] * x + G[k] * (1 - x) for k in G}
    strain = (G["a"] - p["a"]) / p["a"]
    ppz = 2 * strain * (p["e31"] - p["e33"] * p["c13"] / p["c33"])
    return (G["Psp"] - (p["Psp"] + ppz)) / Q / 1e4
for x in (0.15, 0.25, 0.30, 0.35):
    print(f"x = {x:.2f}: n_s = {sheet(x):.3g} cm^-2")
assert abs(sheet(0.25) - 1.39e13) < 5e10 and abs(sheet(0.35) - 1.99e13) < 5e10
xs = np.linspace(0, 0.4, 41)
plt.plot(xs, [sheet(x) for x in xs]); plt.xlabel("Al fraction x"); plt.ylabel("n_s (cm^-2)"); plt.show()

x = 0.15: n_s = 8.21e+12 cm^-2
x = 0.25: n_s = 1.39e+13 cm^-2
x = 0.30: n_s = 1.69e+13 cm^-2
x = 0.35: n_s = 1.99e+13 cm^-2


## Part 4 — Modulation doping

In [5]:
ns = np.logspace(12, 14, 30)
r2deg = 1 / (Q * ns * 1500)
rsi = [1 / (Q * n * mun(n / 50e-7)) for n in ns]          # same electrons in a 50 nm doped Si layer
plt.loglog(ns, r2deg, label="2DEG, 1500 cm2/Vs"); plt.loglog(ns, rsi, label="doped Si, 50 nm")
plt.xlabel("electrons per cm^2"); plt.ylabel("sheet resistance (ohm/sq)"); plt.legend(); plt.show()
ratio = (1 / (Q * 1e13 * mun(2e18))) / (1 / (Q * 1e13 * 1500))
print(f"ratio at 1e13 cm^-2: {ratio:.1f}")
assert abs(ratio - 7.9) < 0.05

ratio at 1e13 cm^-2: 7.9


## Part 5 — Dies per wafer

In [6]:
def gross(d, A_):
    return math.pi * (d / 2) ** 2 / A_ - math.pi * d / math.sqrt(2 * A_)
def counted(d, A_, offsets=9):
    s = math.sqrt(A_); best = 0
    for ox in np.linspace(0, s, offsets, endpoint=False):
        for oy in np.linspace(0, s, offsets, endpoint=False):
            X, Y = np.meshgrid(np.arange(-d / 2 - s + ox, d / 2, s), np.arange(-d / 2 - s + oy, d / 2, s))
            ok = np.all([(cx) ** 2 + (cy) ** 2 <= (d / 2) ** 2 for cx, cy in ((X, Y), (X + s, Y), (X, Y + s), (X + s, Y + s))], axis=0)
            best = max(best, int(ok.sum()))
    return best
for d in (150, 200, 300):
    for A_ in (25, 100):
        print(f"{d} mm, {A_} mm2: formula {gross(d, A_):.0f}, counted {counted(d, A_)}")
assert abs(counted(150, 25) / gross(150, 25) - 1) < 0.03

150 mm, 25 mm2: formula 640, counted 653
150 mm, 100 mm2: formula 143, counted 150
200 mm, 25 mm2: formula 1168, counted 1187
200 mm, 100 mm2: formula 270, counted 278
300 mm, 25 mm2: formula 2694, counted 2715
300 mm, 100 mm2: formula 640, counted 653
